# Flight Delay Prediction - API + ngrok Demo

Este notebook demuestra el empaquetado operativo del modelo de regresion y su publicacion temporal con ngrok desde Google Drive.

Objetivo de la demo:

- Montar `MyDrive/flight-delay-prediction`.
- Verificar que exista `models/flight_delay_model.joblib`.
- Verificar que exista `src/flight_delay_api.py`.
- Levantar la API FastAPI desde `src/flight_delay_api.py`.
- Publicar la API local con ngrok.
- Probar el endpoint `/health` y `/predict` desde una URL publica.

Antes de ejecutar este notebook, corre primero el notebook principal `Flight Delay Prediction EDA ML.ipynb` hasta la celda de empaquetado del modelo de regresion.

In [ ]:
import subprocess
import sys

for package in ["fastapi", "uvicorn", "pyngrok", "requests", "joblib", "pandas", "scikit-learn", "xgboost"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

print("Dependencias listas")

In [ ]:
from pathlib import Path
import os
import sys

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/flight-delay-prediction')
else:
    PROJECT_ROOT = Path.cwd()

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Proyecto activo en: {PROJECT_ROOT}')
print(f'Contenido esperado: src/, models/, data/')

## 1. Validar artefacto del modelo

El archivo `models/flight_delay_model.joblib` debe existir. Ese archivo se genera desde el notebook principal despues de entrenar XGBoost.

In [ ]:
MODEL_PATH = PROJECT_ROOT / "models" / "flight_delay_model.joblib"
API_PATH = PROJECT_ROOT / "src" / "flight_delay_api.py"
FEATURES_PATH = PROJECT_ROOT / "src" / "flight_delay_features.py"

missing_paths = [path for path in [MODEL_PATH, API_PATH, FEATURES_PATH] if not path.exists()]

if missing_paths:
    missing_text = "\n".join(str(path) for path in missing_paths)
    raise FileNotFoundError(
        "Faltan archivos necesarios en Google Drive:\n"
        f"{missing_text}\n\n"
        "Verifica que subiste la carpeta src/ y que ejecutaste el empaquetado del modelo."
    )

print(f"Modelo encontrado: {MODEL_PATH.resolve()}")
print(f"API encontrada: {API_PATH.resolve()}")

## 2. Levantar FastAPI localmente

La API queda escuchando en `http://127.0.0.1:8000`. Se ejecuta en un thread para poder seguir usando el notebook.

In [ ]:
import threading
import time

import requests
import uvicorn

def run_api():
    uvicorn.run("src.flight_delay_api:app", host="0.0.0.0", port=8000, log_level="info")

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

time.sleep(5)

health_response = requests.get("http://127.0.0.1:8000/health", timeout=10)
print(health_response.status_code)
print(health_response.json())

## 3. Publicar con ngrok

Si tu cuenta de ngrok requiere token, define `NGROK_AUTHTOKEN` antes de abrir el tunel.

In [ ]:
import os

from pyngrok import ngrok

NGROK_AUTHTOKEN = os.getenv("NGROK_AUTHTOKEN", "")

if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)

ngrok.kill()
public_url = ngrok.connect(8000, "http").public_url

print("URL publica de la API:")
print(public_url)
print("Health check:", f"{public_url}/health")
print("Prediction endpoint:", f"{public_url}/predict")

## 4. Probar prediccion desde ngrok

Este payload simula un vuelo con senales de riesgo operativo. El consumidor externo debe usar la misma estructura.

In [ ]:
sample_payload = {
    "features": {
        "HOUR_OF_DAY": 17,
        "IS_PEAK_HOUR": 1,
        "IS_WEEKEND": 0,
        "IS_HIGH_SEASON": 1,
        "CASCADING_DELAY_FLAG": 1,
        "ROUTE_AVG_DELAY": 12.4,
        "AIRLINE_AVG_DELAY": 9.8,
        "AIRLINE_ENC": 3,
        "DISTANCE": 740,
        "DEPARTURE_DELAY": 18,
        "MONTH": 7,
        "DAY_OF_WEEK": 5,
        "AIR_SYSTEM_DELAY": 10,
        "WEATHER_DELAY": 0,
    }
}

prediction_response = requests.post(f"{public_url}/predict", json=sample_payload, timeout=10)
print(prediction_response.status_code)
print(prediction_response.json())